# Module 2 — Dropout Risk Classifier + SHAP Explainability
### AI-Powered Student LMS | Phase 3

**Goal:** Classify each student as Dropout / Enrolled / Graduate risk  
**Models:** Random Forest + XGBoost (compare both)  
**Explainability:** SHAP values — explain WHY each student is at risk

---

In [ ]:
# ── CELL 1 ── Imports & Paths ──────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, os, warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, roc_auc_score, roc_curve
)
from sklearn.preprocessing import LabelEncoder, label_binarize
from xgboost import XGBClassifier
import shap

BASE   = r'C:\Users\DELL\Desktop\AI_Student_LMS'
MODELS = os.path.join(BASE, 'models')
ASSETS = os.path.join(BASE, 'assets')
os.makedirs(MODELS, exist_ok=True)
os.makedirs(ASSETS, exist_ok=True)

print('✅ All imports done!')

In [ ]:
# ── CELL 2 ── Load Data WITH Real Feature Names ────────────────
# Load raw data to get real column names
df_raw = pd.read_csv(os.path.join(BASE, 'data', 'raw', 'data.csv'), sep=';')

# Encode target same way as preprocessing
le = LabelEncoder()
df_raw['Target_enc'] = le.fit_transform(df_raw['Target'])
CLASS_NAMES = list(le.classes_)  # ['Dropout', 'Enrolled', 'Graduate']

# Get feature columns (all except Target columns)
feature_cols = [c for c in df_raw.columns if c not in ['Target', 'Target_enc']]
FEATURE_NAMES = feature_cols

# Load preprocessed numpy arrays
X_train = np.load(os.path.join(BASE, 'data', 'processed', 'X_train.npy'))
X_test  = np.load(os.path.join(BASE, 'data', 'processed', 'X_test.npy'))
y_train = np.load(os.path.join(BASE, 'data', 'processed', 'y_train.npy'))
y_test  = np.load(os.path.join(BASE, 'data', 'processed', 'y_test.npy'))

# Wrap in DataFrame with real column names
n_features = X_train.shape[1]
FEATURE_NAMES = FEATURE_NAMES[:n_features]  # match exact count

X_train_df = pd.DataFrame(X_train, columns=FEATURE_NAMES)
X_test_df  = pd.DataFrame(X_test,  columns=FEATURE_NAMES)

# Save feature names for Streamlit later
joblib.dump(FEATURE_NAMES, os.path.join(MODELS, 'feature_names.pkl'))
joblib.dump(le,            os.path.join(MODELS, 'label_encoder.pkl'))

print(f'✅ Data loaded with real feature names!')
print(f'Train: {X_train_df.shape}  |  Test: {X_test_df.shape}')
print(f'Classes: {CLASS_NAMES}')
print(f'Features ({len(FEATURE_NAMES)}): {FEATURE_NAMES[:5]}...')

In [ ]:
# ── CELL 3 ── Train Random Forest ─────────────────────────────
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train_df, y_train)
rf_preds = rf.predict(X_test_df)
rf_proba = rf.predict_proba(X_test_df)

print('=== Random Forest Classification Report ===')
print(classification_report(y_test, rf_preds, target_names=CLASS_NAMES))

rf_acc = accuracy_score(y_test, rf_preds)
rf_roc = roc_auc_score(y_test, rf_proba, multi_class='ovr')
print(f'Accuracy : {rf_acc*100:.2f}%')
print(f'ROC-AUC  : {rf_roc:.4f}')

joblib.dump(rf, os.path.join(MODELS, 'random_forest.pkl'))
print('✅ Random Forest saved!')

In [ ]:
# ── CELL 4 ── Train XGBoost ────────────────────────────────────
xgb = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    eval_metric='mlogloss',
    n_jobs=-1
)
xgb.fit(X_train_df, y_train)
xgb_preds = xgb.predict(X_test_df)
xgb_proba = xgb.predict_proba(X_test_df)

print('=== XGBoost Classification Report ===')
print(classification_report(y_test, xgb_preds, target_names=CLASS_NAMES))

xgb_acc = accuracy_score(y_test, xgb_preds)
xgb_roc = roc_auc_score(y_test, xgb_proba, multi_class='ovr')
print(f'Accuracy : {xgb_acc*100:.2f}%')
print(f'ROC-AUC  : {xgb_roc:.4f}')

joblib.dump(xgb, os.path.join(MODELS, 'xgboost.pkl'))
print('✅ XGBoost saved!')

# Pick best model
best_model  = xgb if xgb_acc >= rf_acc else rf
best_preds  = xgb_preds if xgb_acc >= rf_acc else rf_preds
best_proba  = xgb_proba if xgb_acc >= rf_acc else rf_proba
best_name   = 'XGBoost' if xgb_acc >= rf_acc else 'Random Forest'
print(f'\n🏆 Best model: {best_name}')

In [ ]:
# ── CELL 5 ── Confusion Matrix (Both Models) ──────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, preds, title in zip(axes,
    [rf_preds, xgb_preds],
    ['Random Forest', 'XGBoost']):
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASS_NAMES,
                yticklabels=CLASS_NAMES, ax=ax,
                linewidths=0.5)
    ax.set_title(f'{title} — Confusion Matrix', fontsize=13, fontweight='bold')
    ax.set_xlabel('Predicted Label')
    ax.set_ylabel('Actual Label')

plt.suptitle('Model Comparison — Confusion Matrices', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(ASSETS, 'confusion_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✅ Confusion matrix saved!')

In [ ]:
# ── CELL 6 ── ROC-AUC Curve ───────────────────────────────────
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc

y_test_bin = label_binarize(y_test, classes=[0, 1, 2])
colors = ['#E74C3C', '#F39C12', '#2ECC71']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, proba, title in zip(axes,
    [rf_proba, xgb_proba],
    ['Random Forest', 'XGBoost']):
    for i, (cls, color) in enumerate(zip(CLASS_NAMES, colors)):
        fpr, tpr, _ = roc_curve(y_test_bin[:, i], proba[:, i])
        roc_auc = auc(fpr, tpr)
        ax.plot(fpr, tpr, color=color, lw=2,
                label=f'{cls} (AUC = {roc_auc:.3f})')
    ax.plot([0, 1], [0, 1], 'k--', lw=1)
    ax.set_xlim([0.0, 1.0])
    ax.set_ylim([0.0, 1.05])
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title(f'{title} — ROC Curve', fontsize=13, fontweight='bold')
    ax.legend(loc='lower right')

plt.suptitle('ROC-AUC Curves — Multi-class', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(ASSETS, 'roc_auc_curve.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✅ ROC-AUC curve saved!')

In [ ]:
# ── CELL 7 ── Feature Importance (Real Names) ─────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for ax, model, title in zip(axes,
    [rf, xgb],
    ['Random Forest', 'XGBoost']):
    feat_imp = pd.Series(model.feature_importances_,
                         index=FEATURE_NAMES)
    feat_imp = feat_imp.sort_values(ascending=False)[:15]
    feat_imp.sort_values().plot(kind='barh', ax=ax, color='#378ADD')
    ax.set_title(f'Top 15 Features — {title}', fontsize=13, fontweight='bold')
    ax.set_xlabel('Importance Score')

plt.suptitle('Feature Importance Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(ASSETS, 'feature_importance.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✅ Feature importance saved with real column names!')

In [ ]:
# ── CELL 8 ── SHAP: Global Feature Importance ─────────────────
print('Computing SHAP values... (takes ~30 seconds)')

# Use XGBoost for SHAP (faster & native support)
explainer = shap.TreeExplainer(xgb)

# Use a sample for speed (200 students)
X_sample = X_test_df.sample(200, random_state=42)
shap_values = explainer.shap_values(X_sample)

print('✅ SHAP values computed!')
print(f'SHAP values shape: {np.array(shap_values).shape}')
# Shape: (n_classes, n_samples, n_features)

In [ ]:
# ── CELL 9 ── SHAP Plot 1: Summary (Beeswarm) ─────────────────
# Show SHAP for Dropout class (class 0) — most important for us
plt.figure()
shap.summary_plot(
    shap_values[0],          # class 0 = Dropout
    X_sample,
    feature_names=FEATURE_NAMES,
    plot_type='dot',
    max_display=15,
    show=False
)
plt.title('SHAP Summary — Dropout Risk Factors', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(ASSETS, 'shap_01_summary_dropout.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print('✅ SHAP summary plot saved!')

In [ ]:
# ── CELL 10 ── SHAP Plot 2: Bar Chart (Mean |SHAP|) ───────────
plt.figure()
shap.summary_plot(
    shap_values[0],
    X_sample,
    feature_names=FEATURE_NAMES,
    plot_type='bar',
    max_display=15,
    show=False
)
plt.title('SHAP Feature Importance — Dropout Class', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(ASSETS, 'shap_02_bar_chart.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print('✅ SHAP bar chart saved!')

In [ ]:
# ── CELL 11 ── SHAP Plot 3: Force Plot (Single Student) ───────
# Explain prediction for 1 individual student
student_idx = 0  # Change this to see different students

print(f'Explaining prediction for Student #{student_idx}')
print(f'Actual class    : {CLASS_NAMES[int(y_test[student_idx])]}')
print(f'Predicted class : {CLASS_NAMES[int(xgb_preds[student_idx])]}')

# Force plot for Dropout probability (class 0)
shap.initjs()
force_plot = shap.force_plot(
    explainer.expected_value[0],
    shap_values[0][student_idx],
    X_sample.iloc[student_idx],
    feature_names=FEATURE_NAMES,
    matplotlib=True,
    show=False
)
plt.title(f'SHAP Force Plot — Student #{student_idx} (Dropout Risk)', fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(ASSETS, 'shap_03_force_plot.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print('✅ Force plot saved!')

In [ ]:
# ── CELL 12 ── SHAP Plot 4: Waterfall (Best for Viva Demo) ────
# Waterfall plot = best visual for explaining single student in viva
shap_explanation = shap.Explanation(
    values      = shap_values[0][student_idx],
    base_values = explainer.expected_value[0],
    data        = X_sample.iloc[student_idx].values,
    feature_names = FEATURE_NAMES
)

plt.figure()
shap.waterfall_plot(shap_explanation, max_display=12, show=False)
plt.title(f'SHAP Waterfall — Why Student #{student_idx} is Predicted as Dropout Risk',
          fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(ASSETS, 'shap_04_waterfall.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print('✅ Waterfall plot saved — perfect for viva demo!')

In [ ]:
# ── CELL 13 ── Final Model Comparison Summary ─────────────────
print('=' * 55)
print('FINAL MODEL COMPARISON SUMMARY')
print('=' * 55)
print(f"{'Metric':<25} {'Random Forest':>15} {'XGBoost':>15}")
print('-' * 55)
print(f"{'Accuracy':<25} {rf_acc*100:>14.2f}% {xgb_acc*100:>14.2f}%")
print(f"{'ROC-AUC (OvR)':<25} {rf_roc:>15.4f} {xgb_roc:>15.4f}")
print('-' * 55)
print(f'\n🏆 Best Model Selected: {best_name}')
print(f'\n📁 Saved files:')
print(f'   models/random_forest.pkl')
print(f'   models/xgboost.pkl')
print(f'   models/feature_names.pkl')
print(f'   models/label_encoder.pkl')
print(f'\n📊 SHAP charts saved to assets/:')
print(f'   shap_01_summary_dropout.png')
print(f'   shap_02_bar_chart.png')
print(f'   shap_03_force_plot.png')
print(f'   shap_04_waterfall.png')
print(f'\n✅ MODULE 2 COMPLETE!')
print(f'🚀 Next: 05_exam_score_predictor.ipynb')